# Streaming Responses with the Messages API

Claude can **stream** its response token by token instead of returning the full message at once. Streaming lowers **perceived latency** (text appears as it generates) and lets you process output incrementally. This notebook covers both streaming styles the SDK offers.

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

### 1. Install dependencies

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

`load_dotenv()` reads your **`ANTHROPIC_API_KEY`** from a local `.env` file into the process environment so the SDK can pick it up automatically.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

`Anthropic()` reads the key from the environment. `model` uses the **unversioned alias** so it always resolves to the current Haiku 4.5 snapshot.

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

Small builders keep the message list tidy: `add_user_message` and `add_assistant_message` append correctly-shaped turns, and `chat` wraps a non-streaming `messages.create` call. The streaming demos below reuse `add_user_message`.

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### 5. Stream style A: the raw event iterator

Passing **`stream=True`** to `messages.create` returns an iterator of **typed events** rather than a finished message. This is the **low-level** path: you receive every event and decide how to handle each type.

| Event type | What it carries |
|---|---|
| `message_start` | The opening message shell (id, model, empty content) |
| `content_block_start` / `content_block_delta` / `content_block_stop` | The text chunks, arriving as `delta` pieces |
| `message_delta` | Top-level updates such as `stop_reason` and output-token usage |
| `message_stop` | End of the stream |

The loop below prints **every** event, so expect verbose output. That is the point: it shows the full event sequence you would filter down to `content_block_delta` in production.

In [5]:
messages = []

add_user_message(messages, "Write 1 short story about a dog chasing a cat.")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CcoEXWju4at34cKAUuDda', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=21, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text=' The Great Chase\n\nBiscuit the golden retriever spotted Whiskers the tabby cat sunbathing on the garden', type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text=" fence. For months, they'd been neighbors in an uneasy tr", type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text="uce. But today, something primal stirred in Biscuit's chest.\n\nHe", type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text=" bolted.\n\nWhiskers' eyes snapped open just as a furry blur", type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' thundered toward the fence. With the grace of a gymnast, the cat leaped down and spr', type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text='inted across the yard. Biscuit crashed through the fence gate, his ears flying,', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' his tongue flapping in the wind like a victory flag.\n\nRound the oak tree they', type='text_delta'), index=0, type='content_block_delta')


...

RawContentBlockDeltaEvent(delta=TextDelta(text="eled by pure joy.\n\nThe chase led straight toward Mrs. Henderson's house. Whiskers", type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' darted up a drainpipe. Biscuit skidded to a stop,', type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text=' whining, unable to follow.\n\nPerched safely on the roof', type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text=', Whiskers looked down at the panting dog below. Her tail swished with', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=" what almost looked like amusement. Biscuit's tail wagged just as hard.\n\nMrs", type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text='. Henderson came out with two treats—one for each of them.', type='text_delta'), index=0, type='content_block_delta')


RawContentBlockDeltaEvent(delta=TextDelta(text='\n\nAs Biscuit munched his biscuit and Whiskers licked hers, something passed', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' between them. An understanding, perhaps. Tomorrow, they both knew, they would do it all again.', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockStopEvent(index=0, type='content_block_stop')
RawMessageDeltaEvent(delta=Delta(container=None, stop_details=None, stop_reason='end_turn', stop_sequence=None), type='message_delta', usage=MessageDeltaUsage(cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=21, output_tokens=335, server_tool_use=None))
RawMessageStopEvent(type='message_stop')


### 6. Stream style B: the `messages.stream()` context manager

**`client.messages.stream(...)`** is the **higher-level** path. Used as a `with` block, it hands you a **`.text_stream`** iterator that yields just the text deltas (no event unpacking), and it **accumulates the full message for you** in the background.

> **Gotcha:** the context manager auto-assembles the final message as it streams. Call **`.get_final_message()`** after the `with` block to retrieve the complete `Message` object - id, full `content`, `stop_reason`, and `usage` - without stitching deltas together yourself.

| | Style A: `stream=True` | Style B: `messages.stream()` |
|---|---|---|
| Level | Low - raw typed events | High - text deltas + auto-assembly |
| You handle | Every event type | Just the text (optional) |
| Final message | Rebuild from events | `.get_final_message()` |
| Best for | Custom event routing, tool-use streaming | Printing text to a UI fast |

The loop below consumes `.text_stream` (printing is commented out for a quiet demo), then prints the assembled final message.

In [6]:
messages = []

add_user_message(messages, "Write 1 short story about a dog chasing a cat.")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()

ParsedMessage(id='msg_011CcoEXqKbSi8enNnxtTdBg', container=None, content=[ParsedTextBlock(citations=None, text="# The Chase\n\nBiscuit the golden retriever caught the scent first—that unmistakable smell of Whiskers, the orange tabby from three gardens over.\n\nShe was sunbathing on the fence post, utterly unbothered, when Biscuit bounded around the corner at full speed. The moment their eyes met, Whiskers yawned, stretched, and leaped down into the neighbor's yard with the grace of a tiny acrobat.\n\nWhat followed was pure chaos.\n\nBiscuit crashed through the hedge, yelping with joy, convinced this was the greatest game ever invented. Whiskers weaved between rosebushes, leaped over a birdbath, and scrambled up the oak tree with three bounds. She settled on a branch, tail swishing smugly.\n\nBiscuit skidded to a stop at the tree's base, panting heavily, tail still wagging. She barked twice—a happy, confused sound—then began running circles around the trunk, as if this might somehow mak